In [1]:
#  1: نقرأ ملف الإكسل الأصلي 
import pandas as pd

input_file = "trainData.xlsx" 
all_sheets = pd.read_excel(input_file, sheet_name=None)

print("Sheets:", list(all_sheets.keys()))


Sheets: ['Gemini', 'ChatGPT', 'Tele', 'Telegram', 'DeepSeek']


In [2]:
#  2: نوحّد التصنيفات داخل كل شيت  
# االقضايا الأسرية" نخليه "قضايا الأحوال الشخصية

FAMILY = "القضايا الأسرية"
PERSONAL = "قضايا الأحوال الشخصية"

updated_sheets = {}

for sheet_name, sheet_df in all_sheets.items():
    if sheet_df is None or sheet_df.empty:
        updated_sheets[sheet_name] = sheet_df
        continue

    df2 = sheet_df.copy()

    # نتأكد أن عمود التصنيف موجود
    if "التصنيف" in df2.columns:
        df2["التصنيف"] = df2["التصنيف"].astype(str).str.strip()
        df2.loc[df2["التصنيف"] == FAMILY, "التصنيف"] = PERSONAL

    updated_sheets[sheet_name] = df2

print("Done: labels unified in-memory.")


Done: labels unified in-memory.


In [3]:
#  3: نطلع أرقام قبل/بعد للتأكد 
#    نتأكد أن "القضايا الأسرية"  صارت "أحوال شخصية

def count_labels(sheets_dict):
    counts = {}
    for sname, sdf in sheets_dict.items():
        if sdf is None or sdf.empty or "التصنيف" not in sdf.columns:
            continue
        vc = sdf["التصنيف"].astype(str).str.strip().value_counts()
        counts[sname] = vc
    return counts

before_counts = count_labels(all_sheets)
after_counts = count_labels(updated_sheets)

print("مثال قبل (Gemini):")
print(before_counts.get("Gemini", pd.Series(dtype=int)).head(20))

print("\nمثال بعد (Gemini):")
print(after_counts.get("Gemini", pd.Series(dtype=int)).head(20))


مثال قبل (Gemini):
التصنيف
القضايا الجنائية         154
القضايا الأسرية          125
قضايا الأحوال الشخصية    101
القضايا العقارية          98
القضايا العمالية          97
القضايا الإدارية          88
القضايا المالية           88
القضايا التجارية          86
Name: count, dtype: int64

مثال بعد (Gemini):
التصنيف
قضايا الأحوال الشخصية    226
القضايا الجنائية         154
القضايا العقارية          98
القضايا العمالية          97
القضايا الإدارية          88
القضايا المالية           88
القضايا التجارية          86
Name: count, dtype: int64


In [4]:
#  4: نحفظ النسخة الجديدة
output_file = "trainData_merged_personal_status_v1.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet_name, sheet_df in updated_sheets.items():
        if sheet_df is None:
            continue
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Saved:", output_file)


Saved: trainData_merged_personal_status_v1.xlsx


In [5]:
#  5: نتأكد من الملف الجديد أنه انحفظ صح ونقرأ منه 
check_sheets = pd.read_excel(output_file, sheet_name=None)
print("New file sheets:", list(check_sheets.keys()))

# نعرض 8 صفوف من شيت Gemini للتأكد
check_sheets["Gemini"][["نص الاستشارة", "التصنيف"]].head(8)


New file sheets: ['Gemini', 'ChatGPT', 'Tele', 'Telegram', 'DeepSeek']


,نص الاستشارة,التصنيف
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,القضايا التجارية
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,القضايا العمالية
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,القضايا الجنائية
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,قضايا الأحوال الشخصية
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,القضايا العقارية
5,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,القضايا الإدارية
6,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه.,القضايا المالية
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,قضايا الأحوال الشخصية


In [7]:
#  6: نجمع كل الشيتات من الملف الجديد في جدول واحد للتحليل

dfs_new = []

for sheet_name, sheet_df in check_sheets.items():
    if sheet_df is None or sheet_df.empty:
        continue
    
    temp = sheet_df.copy()
    temp["source"] = sheet_name  # نضيف عمود يوضح مصدر البيانات
    dfs_new.append(temp)

df_new = pd.concat(dfs_new, ignore_index=True)

print("Total rows in new file:", df_new.shape[0])
print("Total columns:", df_new.shape[1])


Total rows in new file: 2321
Total columns: 3


In [8]:
#  7: نطبع جميع التصنيفات الموجودة وعدد الاستشارات في كل تصنيف

LABEL_COL = "التصنيف"  

df_new[LABEL_COL] = df_new[LABEL_COL].astype(str).str.strip()

counts = df_new[LABEL_COL].value_counts().sort_values(ascending=False)

print("All classifications and their counts:\n")
print(counts)

print("\nNumber of unique classes:", counts.shape[0])


All classifications and their counts:

التصنيف
القضايا العمالية         458
قضايا الأحوال الشخصية    437
القضايا العقارية         327
القضايا الإدارية         291
القضايا التجارية         278
القضايا الجنائية         271
القضايا المالية          259
Name: count, dtype: int64

Number of unique classes: 7


In [9]:
#  8: نتأكد أن القضايا الأسرية مو موجودة

if "القضايا الأسرية" in counts.index:
    print(" لا زالت القضايا الأسرية موجودة وعددها:", counts["القضايا الأسرية"])
else:
    print(" تم دمج القضايا الأسرية بالكامل في قضايا الأحوال الشخصية")


 تم دمج القضايا الأسرية بالكامل في قضايا الأحوال الشخصية


In [10]:
#  نعيد حفظ الملف باسم اخر 

academic_name = "Legal_Consultations_Standardized.xlsx"

with pd.ExcelWriter(academic_name, engine="openpyxl") as writer:
    for sheet_name, sheet_df in check_sheets.items():
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)

print("File saved as:", academic_name)


File saved as: Legal_Consultations_Standardized.xlsx


In [11]:
# نتأكد أن الملف انحفظ و فيه نفس الشيتات

verify = pd.read_excel(academic_name, sheet_name=None)
print("Sheets inside academic file:", list(verify.keys()))


Sheets inside academic file: ['Gemini', 'ChatGPT', 'Tele', 'Telegram', 'DeepSeek']
